# Assignment 2.6 - Infer Flux & Qwen-Image (Diffusers) + ComfyUI

Hai model flow-matching/DiT cap SOTA, dung qua **Diffusers** cho ca **Text2Img** va **Img2Img (Image Editing)**.

> **Yeu cau phan cung:** day la model rat lon (Flux ~12B, Qwen-Image ~20B).
> Nen chay tren GPU >= 24GB, dung `bfloat16` + `enable_model_cpu_offload()`.
> Tren GPU nho hon co the dung ban quantized (GGUF/4-bit) trong ComfyUI.

Anh luu vao `output/`.

In [4]:
!pip uninstall -y torchao
# Dam bao diffusers du moi cho Flux/Qwen-Image:
!pip install -U "diffusers>=0.35.0" "transformers>=4.46.0" accelerate safetensors sentencepiece


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
# pip install -U diffusers transformers accelerate safetensors sentencepiece
import os, torch
from diffusers.utils import load_image

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32
print("device:", device, "| dtype:", dtype)

# Diffusers 0.38 registers a custom op with postponed annotations;
# torch 2.4.x cannot infer that schema directly, so resolve the hints before importing Flux/Qwen.
import typing
import torch._library.infer_schema as _infer_schema_mod

_orig_infer_schema = _infer_schema_mod.infer_schema

def _patched_infer_schema(prototype_function, mutates_args):
    try:
        hints = typing.get_type_hints(
            prototype_function,
            globalns=getattr(prototype_function, "__globals__", {}),
        )
    except Exception:
        hints = {}
    if hints:
        original_annotations = dict(getattr(prototype_function, "__annotations__", {}))
        prototype_function.__annotations__ = {**original_annotations, **hints}
        try:
            return _orig_infer_schema(prototype_function, mutates_args)
        finally:
            prototype_function.__annotations__ = original_annotations
    return _orig_infer_schema(prototype_function, mutates_args)

_infer_schema_mod.infer_schema = _patched_infer_schema

try:
    import torch._custom_op.impl as _custom_impl
    _custom_impl.infer_schema = _patched_infer_schema
except Exception:
    pass

print("patched torch schema inference for diffusers import")

device: cuda | dtype: torch.bfloat16
patched torch schema inference for diffusers import


In [ ]:
# === XU LY 2 NGUYEN NHAN KHIEN FLUX/QWEN KHONG CHAY ===

# (1) Flux la repo GATED -> phai dang nhap HF token VA da bam "Agree" license tai:
#     https://huggingface.co/black-forest-labs/FLUX.1-dev   (hoac .../FLUX.1-schnell)
from huggingface_hub import login
login()                       # dan token tu https://huggingface.co/settings/tokens
# login(token="hf_xxx")       # hoac dien thang token o day

# (2) Dung luong dia: Flux ~24GB, Qwen-Image ~40GB. Kiem tra cho trong:
import shutil
free_gb = shutil.disk_usage("/root").free / 1e9
print(f"Free disk: {free_gb:.1f} GB")
if free_gb < 30:
    print("!! Thieu dia. Don cache cu (vd SD-1.5/CLIP) roi chay lai:")
    print("   !rm -rf /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5")
    print("   Hoac dung runtime GPU lon hon (A100) co nhieu dia/VRAM hon.")

# Model dung chung cho cac cell Flux ben duoi:
#   FLUX.1-dev    : chat luong cao  (guidance 3.5, ~50 steps) - GATED
#   FLUX.1-schnell: nhanh, nhe hon  (guidance 0.0, ~4 steps)
FLUX_MODEL = "black-forest-labs/FLUX.1-dev"

## 1) FLUX.1 - Text2Img

Dung `FluxPipeline` voi `black-forest-labs/FLUX.1-dev` (hoac `FLUX.1-schnell` cho nhanh, 4 buoc).

In [ ]:
from diffusers import FluxPipeline

# Load Flux (dung FLUX_MODEL da chon o cell setup)
pipe = FluxPipeline.from_pretrained(FLUX_MODEL, torch_dtype=dtype)
pipe.enable_model_cpu_offload()          # tiet kiem VRAM (chay duoc ~16GB GPU, cham hon)

prompt = "a cinematic photo of a small cabin in a snowy forest at sunset, highly detailed"
image = pipe(
    prompt,
    height=1024, width=1024,
    guidance_scale=3.5,                  # FLUX.1-dev: 3.5  |  FLUX.1-schnell: 0.0
    num_inference_steps=50,              # FLUX.1-dev: ~50  |  FLUX.1-schnell: ~4
    max_sequence_length=512,             # FLUX.1-schnell: 256
    generator=torch.Generator("cpu").manual_seed(0),
).images[0]

image.save(os.path.join(OUTPUT_DIR, "flux_t2i.png"))
print("saved output/flux_t2i.png")
image

## 2) FLUX.1 - Img2Img & Image Editing

- **Img2Img:** `FluxImg2ImgPipeline` (giu bo cuc anh goc, ve lai theo prompt, dieu khien bang `strength`).
- **Image Editing (chinh sua theo lenh):** dung `FluxKontextPipeline` voi `FLUX.1-Kontext-dev`.

In [ ]:
from diffusers import FluxImg2ImgPipeline

pipe_i2i = FluxImg2ImgPipeline.from_pretrained(FLUX_MODEL, torch_dtype=dtype)
pipe_i2i.enable_model_cpu_offload()

init_image = load_image(
    "https://raw.githubusercontent.com/CompVis/stable-diffusion/main/assets/stable-samples/img2img/sketch-mountains-input.jpg"
).resize((1024, 1024))

image = pipe_i2i(
    prompt="a fantasy landscape, vivid autumn colors, highly detailed",
    image=init_image,
    strength=0.6,                  # 0 = giu nguyen anh goc, 1 = ve moi hoan toan
    guidance_scale=3.5,
    num_inference_steps=40,
    generator=torch.Generator("cpu").manual_seed(0),
).images[0]

image.save(os.path.join(OUTPUT_DIR, "flux_i2i.png"))
print("saved output/flux_i2i.png")
image

In [ ]:
# (Tuy chon) Image Editing bang lenh voi FLUX.1-Kontext
# from diffusers import FluxKontextPipeline
# kontext = FluxKontextPipeline.from_pretrained("black-forest-labs/FLUX.1-Kontext-dev", torch_dtype=dtype)
# kontext.enable_model_cpu_offload()
# edited = kontext(image=init_image, prompt="make it winter with snow", guidance_scale=2.5).images[0]
# edited.save(os.path.join(OUTPUT_DIR, "flux_kontext_edit.png"))
print("Kontext: bo comment de chinh sua anh theo lenh (instruction editing).")

## 3) Qwen-Image - Text2Img

In [ ]:
from diffusers import DiffusionPipeline

qwen = DiffusionPipeline.from_pretrained("Qwen/Qwen-Image", torch_dtype=dtype)
qwen.enable_model_cpu_offload()

prompt = "a coffee shop storefront with a neon sign that reads \"Flow Matching\", photorealistic"
img = qwen(
    prompt,
    width=1024, height=1024,
    num_inference_steps=50,
    true_cfg_scale=4.0,
    generator=torch.Generator("cpu").manual_seed(0),
).images[0]
img.save(os.path.join(OUTPUT_DIR, "qwen_t2i.png"))
print("saved output/qwen_t2i.png")
img

## 4) Qwen-Image-Edit - Img2Img / Image Editing

Qwen-Image-Edit manh ve **chinh sua theo lenh** (them/bot vat the, doi chu, doi phong cach).

In [ ]:
from diffusers import QwenImageEditPipeline

qwen_edit = QwenImageEditPipeline.from_pretrained("Qwen/Qwen-Image-Edit", torch_dtype=dtype)
qwen_edit.enable_model_cpu_offload()

edit_in = load_image(
    "https://raw.githubusercontent.com/CompVis/stable-diffusion/main/assets/stable-samples/img2img/sketch-mountains-input.jpg"
).resize((1024, 1024))

img = qwen_edit(
    image=edit_in,
    prompt="add a full moon in the sky and make it nighttime",
    num_inference_steps=40,
    true_cfg_scale=4.0,
    generator=torch.Generator("cpu").manual_seed(0),
).images[0]
img.save(os.path.join(OUTPUT_DIR, "qwen_edit.png"))
print("saved output/qwen_edit.png")
img

## 5) ComfyUI - Workflow cho Flux & Qwen-Image

ComfyUI da co san template chuan (menu **Workflow -> Browse Templates -> Flux / Qwen-Image**),
day la diem khoi dau tin cay nhat. So do node co ban:

### Flux (Text2Img)
```
UNETLoader(flux1-dev.safetensors)            --MODEL-->
DualCLIPLoader(clip_l + t5xxl, type=flux)    --CLIP--> CLIPTextEncode(prompt)
VAELoader(ae.safetensors)                    --VAE-->
[MODEL]+[CONDITIONING] -> BasicGuider/SamplerCustomAdvanced (hoac KSampler)
EmptySD3LatentImage(1024x1024) -> sampler -> VAEDecode -> SaveImage
```
- **Img2Img:** thay `EmptySD3LatentImage` bang `LoadImage -> VAEEncode` va ha `denoise` (~0.5-0.7).
- **Editing:** dung **FLUX.1-Kontext** node (ReferenceLatent / Kontext) de chinh sua theo lenh.

### Qwen-Image
- Dung node **Qwen-Image** trong template; **Qwen-Image-Edit** cho img2img/editing
  (node nhan anh dau vao + prompt chinh sua).

### Node tang CHAT LUONG
- **Upscale:** `UpscaleModelLoader` + `ImageUpscaleWithModel` (4x-UltraSharp), hoac hi-res 2 pass.
- Negative/true-CFG hop ly; tang steps cho ban `-dev`.

### Node tang TOC DO
- Ban **FLUX.1-schnell** (4 buoc) hoac **GGUF/fp8 quantized** (UnetLoaderGGUF) -> chay duoc GPU nho.
- **TeaCache / WaveSpeed** node de cache buoc -> nhanh hon nhieu.
- Sampler `euler` + scheduler `simple`/`sgm_uniform`, giam steps.
- Bat `--fast` / sage-attention khi khoi dong ComfyUI.

> File workflow JSON nen export truc tiep tu ComfyUI sau khi rap (nut **Save**),
> vi JSON Flux/Qwen rat dai va phu thuoc phien ban node.